# Machine Learning for IT Application Development — All Experiments

This notebook keeps Experiments 01–10 in separate sections so the code does not have to be copied into separate files repeatedly. The structure follows the simplified lab manual. Viva questions are intentionally omitted.

## Dataset folder
Keep all downloaded datasets inside a folder named `datasets` next to this notebook.

| Experiment | Dataset file | Source |
|---|---|---|
| 01 | `student_placement.csv` | Kaggle — Student Placement Prediction Dataset 2026 |
| 02 | `play_tennis.csv` | Kaggle — Play Tennis Dataset |
| 03 | `titanic_train.csv` | Kaggle — Titanic: Machine Learning from Disaster (`train.csv`) |
| 04 | `house_train.csv` | Kaggle — House Prices: Advanced Regression Techniques (`train.csv`) |
| 05 | `titanic_train.csv` | Same Titanic training CSV as Experiment 03 |
| 06 | `Iris.csv` | Kaggle — Iris Species |
| 07 | `Iris.csv` | Same Iris CSV as Experiment 06 |
| 08 | `Iris.csv` | Same Iris CSV as Experiment 06 |
| 09 | `Iris.csv` | Same Iris CSV as Experiment 06 |
| 10 | `data.csv` | Kaggle — Breast Cancer Wisconsin (Diagnostic) Data Set |

For Kaggle files, download the CSV once and reuse it wherever the same dataset appears.

## Setup
Run the following cell once before starting the experiments. The experiment sections reuse these imports and helper paths instead of repeating the same setup code.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path('datasets')
DATA.mkdir(exist_ok=True)

# Main ML libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

# Statistics
from scipy import stats
import statsmodels.api as sm

print('Dataset folder:', DATA.resolve())

### Optional installation
If a library is missing in Jupyter/Colab, install it once. TensorFlow can be relatively large, so install it only when required for Experiments 01 and 10.

In [ ]:
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy statsmodels nltk tensorflow

# Experiment No. 01
## Case study on different Python tools and libraries used in ML

**Aim:** To explore and understand key Python libraries commonly used in machine learning, with practical examples demonstrating their usage.

**Dataset:** Kaggle Student Placement Prediction Dataset 2026. Save its CSV as `datasets/student_placement.csv`.

The manual uses NumPy, Pandas, Matplotlib, Seaborn, scikit-learn, TensorFlow/Keras, NLTK, SciPy and Statsmodels.

In [ ]:
df1 = pd.read_csv(DATA / 'student_placement.csv')
print(df1.head())
print(df1.columns.tolist())

### 1. NumPy — mean CGPA and median salary

In [ ]:
placed = df1[df1['placement_status'] == 'Placed']
print('Mean CGPA:', np.mean(placed['cgpa']))
print('Median Salary:', np.median(placed['salary_package_lpa'].dropna()))

### 2. Pandas — preview and filter

In [ ]:
print(df1.head())
print(df1[(df1['gender'] == 'Female') & (df1['cgpa'] > 8)]
      [['student_id', 'gender', 'cgpa', 'branch']].head())

### 3. Matplotlib — salary distribution

In [ ]:
plt.hist(df1['salary_package_lpa'].dropna(), bins=10)
plt.xlabel('Salary (LPA)')
plt.ylabel('Number of Students')
plt.title('Salary Distribution')
plt.show()

### 4. Seaborn — placement count by branch

In [ ]:
sns.countplot(x='branch', hue='placement_status', data=df1)
plt.xticks(rotation=45)
plt.show()

### 5. scikit-learn — logistic regression for placement

In [ ]:
from sklearn.linear_model import LogisticRegression

d = df1[['cgpa', 'coding_skill_score', 'internships_count', 'placement_status']].dropna()
X = d[['cgpa', 'coding_skill_score', 'internships_count']]
y = d['placement_status'].map({'Not Placed': 0, 'Placed': 1})

X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X, y, test_size=0.2, random_state=42)

model1 = LogisticRegression(max_iter=200)
model1.fit(X_train1, y_train1)
print('Accuracy:', accuracy_score(y_test1, model1.predict(X_test1)))

### 6. TensorFlow / Keras — small neural network

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

net1 = Sequential([
    Dense(8, activation='relu', input_shape=(3,)),
    Dense(1, activation='sigmoid')
])
net1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
net1.fit(X_train1, y_train1, epochs=10, verbose=0)
print('Test accuracy:', net1.evaluate(X_test1, y_test1, verbose=0)[1])

### 7. NLTK — tokenization

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
text = 'The placement training was useful.'
print(word_tokenize(text))

### 8. SciPy — t-test

In [ ]:
placed_cgpa = df1[df1['placement_status'] == 'Placed']['cgpa'].dropna()
not_placed_cgpa = df1[df1['placement_status'] == 'Not Placed']['cgpa'].dropna()
t, p = stats.ttest_ind(placed_cgpa, not_placed_cgpa)
print('t-statistic:', round(t, 3))
print('p-value:', round(p, 3))

### 9. Statsmodels — logistic regression summary

In [ ]:
d = df1[['cgpa', 'coding_skill_score', 'placement_status']].dropna()
X_sm = sm.add_constant(d[['cgpa', 'coding_skill_score']])
y_sm = d['placement_status'].map({'Not Placed': 0, 'Placed': 1})
print(sm.Logit(y_sm, X_sm).fit().summary())

# Experiment No. 02
## FIND-S and Candidate-Elimination concept learning

**Aim:** Write code for the FIND-S algorithm to find the most specific hypothesis from training data read from a CSV file.

**Dataset:** Kaggle Play Tennis Dataset. Save the relevant CSV as `datasets/play_tennis.csv`.

The example in the manual uses six attributes and a final `Goes`/target column. FIND-S uses only positive examples and replaces conflicting attribute values with `?`.

### Manual example — final hypothesis

In [ ]:
print('Final maximally specific hypothesis: <Sunny, Warm, ?, Strong, ?, ?>')

### FIND-S implementation

In [ ]:
df2 = pd.read_csv(DATA / 'play_tennis.csv')

attributes = df2.columns[:-1]
target = df2.columns[-1]
hypothesis = ['0'] * len(attributes)

for _, row in df2.iterrows():
    if str(row[target]).lower() == 'yes':
        if hypothesis == ['0'] * len(attributes):
            hypothesis = list(row[attributes])
        else:
            for i in range(len(attributes)):
                if hypothesis[i] != row[attributes[i]]:
                    hypothesis[i] = '?'

print('Final Hypothesis:', hypothesis)

# Experiment No. 03
## Data Handling with Python

**Aim:** To understand and perform data handling using Python libraries for preprocessing datasets in Machine Learning.

**Dataset:** Kaggle Titanic — Machine Learning from Disaster. Use `train.csv` and save/rename it as `datasets/titanic_train.csv`.

In [ ]:
df3 = pd.read_csv(DATA / 'titanic_train.csv')
print(df3.head())

### Check and clean the data

In [ ]:
print(df3.isnull().sum())
df3['Age'] = df3['Age'].fillna(df3['Age'].median())
df3 = df3.drop_duplicates()

### Transform the data

In [ ]:
df3['Sex'] = df3['Sex'].map({'male': 0, 'female': 1})
df3['FamilySize'] = df3['SibSp'] + df3['Parch'] + 1
print(df3[['Age', 'Sex', 'FamilySize']].head())

### Basic analysis and visualization

In [ ]:
print('Mean age:', df3['Age'].mean())
print('Median age:', df3['Age'].median())

sns.histplot(df3['Age'], kde=True)
plt.title('Age Distribution')
plt.show()

sns.countplot(x='Sex', data=df3)
plt.title('Gender Count')
plt.show()

In [ ]:
df3.to_csv(DATA / 'cleaned_titanic.csv', index=False)
print('Saved:', DATA / 'cleaned_titanic.csv')

# Experiment No. 04
## Linear and Multiple Linear Regression

**Aim:** To implement Linear Regression and Multiple Linear Regression using Python and evaluate their performance.

**Dataset:** Kaggle House Prices — Advanced Regression Techniques. Save its `train.csv` as `datasets/house_train.csv`.

Simple regression uses `GrLivArea` to predict `SalePrice`. Multiple regression uses `GrLivArea`, `OverallQual` and `GarageCars`.

### Load data

In [ ]:
df4 = pd.read_csv(DATA / 'house_train.csv')
data4 = df4[['GrLivArea', 'OverallQual', 'GarageCars', 'SalePrice']].dropna()
print(data4.head())

### Simple Linear Regression

In [ ]:
X = data4[['GrLivArea']]
y = data4['SalePrice']

X_train4, X_test4, y_train4, y_test4 = train_test_split(
    X, y, test_size=0.2, random_state=42)

model4 = LinearRegression()
model4.fit(X_train4, y_train4)
y_pred4 = model4.predict(X_test4)

print('MSE:', mean_squared_error(y_test4, y_pred4))
print('R2:', r2_score(y_test4, y_pred4))

plt.scatter(X_test4, y_test4)
plt.plot(X_test4, y_pred4)
plt.xlabel('Living Area')
plt.ylabel('Sale Price')
plt.show()

### Multiple Linear Regression

In [ ]:
X = data4[['GrLivArea', 'OverallQual', 'GarageCars']]

X_train4, X_test4, y_train4, y_test4 = train_test_split(
    X, y, test_size=0.2, random_state=42)

model4.fit(X_train4, y_train4)
y_pred4 = model4.predict(X_test4)

print('MSE:', mean_squared_error(y_test4, y_pred4))
print('R2:', r2_score(y_test4, y_pred4))

# Experiment No. 05
## Logistic Regression

**Aim:** To implement Logistic Regression for binary classification and evaluate its performance.

**Dataset:** Same Kaggle Titanic `train.csv` used in Experiment 03.

In [ ]:
df5 = pd.read_csv(DATA / 'titanic_train.csv')
df5 = df5[['Age', 'Fare', 'Sex', 'Survived']].dropna()
df5['Sex'] = df5['Sex'].map({'male': 0, 'female': 1})

X = df5[['Age', 'Fare', 'Sex']]
y = df5['Survived']

X_train5, X_test5, y_train5, y_test5 = train_test_split(
    X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LogisticRegression
model5 = LogisticRegression()
model5.fit(X_train5, y_train5)
y_pred5 = model5.predict(X_test5)

print('Accuracy:', accuracy_score(y_test5, y_pred5))

### Confusion matrix and probability plot

In [ ]:
cm = confusion_matrix(y_test5, y_pred5)
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

y_prob = model5.predict_proba(X_test5)[:, 1]
plt.hist(y_prob, bins=10)
plt.xlabel('Probability')
plt.ylabel('Count')
plt.title('Predicted Survival Probability')
plt.show()

# Experiment No. 06
## K-Nearest Neighbour (KNN) Algorithm

**Aim:** To implement and demonstrate KNN for classification.

**Dataset:** Kaggle Iris Species. Save `Iris.csv` as `datasets/Iris.csv`.

In [ ]:
df6 = pd.read_csv(DATA / 'Iris.csv')
X = df6[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']]
y = df6['Species']

X_train6, X_test6, y_train6, y_test6 = train_test_split(
    X, y, test_size=0.2, random_state=42)

from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train6, y_train6)
y_pred6 = knn.predict(X_test6)

print('Accuracy:', accuracy_score(y_test6, y_pred6))
print('Confusion Matrix:\n', confusion_matrix(y_test6, y_pred6))

### 2D decision regions

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
X2 = X[['SepalLengthCm', 'SepalWidthCm']]
knn2 = KNeighborsClassifier(n_neighbors=3).fit(X2, y)
DecisionBoundaryDisplay.from_estimator(knn2, X2, alpha=0.25)
plt.scatter(X2.iloc[:, 0], X2.iloc[:, 1], c=pd.Categorical(y).codes)
plt.show()

# Experiment No. 07
## Feature Selection and Feature Extraction

**Aim:** To implement feature selection and feature extraction techniques for dimensionality reduction.

**Dataset:** Same Kaggle Iris `Iris.csv` used in Experiment 06.

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

# Use non-negative Iris feature values for Chi-Square
selector = SelectKBest(chi2, k=2)
selector.fit(X, pd.Categorical(y).codes)
print('Selected feature indices:', selector.get_support(indices=True))

rfe = RFE(LogisticRegression(max_iter=200), n_features_to_select=2)
rfe.fit(X, pd.Categorical(y).codes)
print('Selected features:', rfe.support_)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
print('Explained variance:', pca.explained_variance_ratio_)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=pd.Categorical(y).codes)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Feature Extraction')
plt.show()

# Experiment No. 08
## Support Vector Machine (SVM) Algorithm

**Aim:** To implement SVM for classification using linear and non-linear kernels.

**Dataset:** Same Kaggle Iris `Iris.csv`.

In [ ]:
from sklearn.svm import SVC

X_train8, X_test8, y_train8, y_test8 = train_test_split(
    X, y, test_size=0.2, random_state=42)

linear = SVC(kernel='linear')
rbf = SVC(kernel='rbf')

linear.fit(X_train8, y_train8)
rbf.fit(X_train8, y_train8)

pred_linear = linear.predict(X_test8)
pred_rbf = rbf.predict(X_test8)

print('Linear accuracy:', accuracy_score(y_test8, pred_linear))
print('RBF accuracy:', accuracy_score(y_test8, pred_rbf))
print('RBF confusion matrix:\n', confusion_matrix(y_test8, pred_rbf))

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
X2 = X[['SepalLengthCm', 'SepalWidthCm']]
svm2 = SVC(kernel='linear').fit(X2, y)
DecisionBoundaryDisplay.from_estimator(svm2, X2, alpha=0.25)
plt.scatter(X2.iloc[:, 0], X2.iloc[:, 1], c=pd.Categorical(y).codes)
plt.show()

# Experiment No. 09
## Decision Tree (CART) Algorithm

**Aim:** To implement and demonstrate the working of the Decision Tree (CART) algorithm for classification.

**Dataset:** Same Kaggle Iris `Iris.csv`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

X_train9, X_test9, y_train9, y_test9 = train_test_split(
    X, y, test_size=0.2, random_state=42)

tree = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
tree.fit(X_train9, y_train9)
y_pred9 = tree.predict(X_test9)

entropy_tree = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
entropy_tree.fit(X_train9, y_train9)

print('Gini accuracy:', accuracy_score(y_test9, y_pred9))
print('Entropy accuracy:', accuracy_score(y_test9, entropy_tree.predict(X_test9)))
print('Confusion Matrix:\n', confusion_matrix(y_test9, y_pred9))

In [ ]:
plt.figure(figsize=(10, 6))
plot_tree(tree, feature_names=X.columns, class_names=tree.classes_, filled=True)
plt.show()

# Experiment No. 10
## Case Study on Deep Learning

**Aim / Case Study:** Use a small neural network for breast-cancer classification.

**Dataset:** Kaggle Breast Cancer Wisconsin (Diagnostic) Data Set. Save the Kaggle `data.csv` as `datasets/data.csv`.

The dataset is tabular: the features are measurements computed from digitized breast-mass images, so the lab uses a dense neural network rather than manually reshaping rows into images.

In [ ]:
df10 = pd.read_csv(DATA / 'data.csv')

X10 = df10.drop(columns=['id', 'diagnosis', 'Unnamed: 32'])
y10 = (df10['diagnosis'] == 'M').astype(int)

X_train10, X_test10, y_train10, y_test10 = train_test_split(
    X10, y10, test_size=0.2, random_state=42, stratify=y10)

from sklearn.preprocessing import StandardScaler
scaler10 = StandardScaler()
X_train10 = scaler10.fit_transform(X_train10)
X_test10 = scaler10.transform(X_test10)

print(X_train10.shape, X_test10.shape)

### Small neural network

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

model10 = Sequential([
    Dense(16, activation='relu', input_shape=(X_train10.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model10.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model10.fit(X_train10, y_train10, epochs=20, batch_size=16, validation_split=0.2, verbose=0)

### Model Evaluation and Results

In [ ]:
test_loss10, test_accuracy10 = model10.evaluate(X_test10, y_test10, verbose=0)
y_pred10 = (model10.predict(X_test10, verbose=0) > 0.5).astype(int).ravel()

print('Test Accuracy:', round(test_accuracy10, 3))
print('Test Loss:', round(test_loss10, 3))

from sklearn.metrics import classification_report
print(classification_report(y_test10, y_pred10))

cm10 = confusion_matrix(y_test10, y_pred10)
plt.imshow(cm10)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.colorbar()
plt.show()

## Final note
Run each experiment section from top to bottom. Experiments 03 and 05 share the Titanic CSV, while Experiments 06–09 share the same Iris CSV. This avoids downloading the same dataset four times.